## CAP182 SS3 – Part B


**Business problem:** predict whether a customer will become dormant within six months, so STADIOEquities can prioritise timely customer-retention interventions.

**Prediction date:** 1 January 2025. Features must represent information available on or before this date. The target is `dormant_within_6m` from `table9_target_dormancy.csv`.

This notebook compares logistic regression and random forest as classification methods and selects the model using held-out test performance. The preprocessing is fitted inside each modelling pipeline, after the train/test split, to prevent test-set information leaking into training.

> Important: tables 2, 3 and 4 contain aggregates measured at the later extract date as well as event timestamps. The later aggregates are deliberately not used. Event-level records are filtered to timestamps on or before 2025-01-01 before customer-level features are calculated.

## 1. Dataset loading

The client extract contains ten CSV tables. The target table contains **3,218 customer records**. In the supplied extract, 1,021 customers (31.7%) are labelled dormant within six months and 2,197 (68.3%) are labelled not dormant. This is a moderately imbalanced binary classification problem, so accuracy will be reported alongside precision, recall, F1-score and ROC-AUC.

Place the extracted `stadioequities` folder beside this notebook, or update `DATA_DIR` below.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

OBSERVATION_DATE = pd.Timestamp("2025-01-01")
DATA_DIR_CANDIDATES = [
    Path("stadioequities"),
    Path("mogalelebogang_client_data/stadioequities"),
    Path("mogalelebogang_client_data") / "stadioequities",
    Path("data/stadioequities"),
]

DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the stadioequities folder. Extract mogalelebogang_client_data.zip "
        "and set DATA_DIR to the folder containing table1_account_onboarding.csv through table10_data_quality_metadata.csv."
    )

def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

account = load_table("table1_account_onboarding.csv")
funding = load_table("table2_funding_balance.csv")
behaviour = load_table("table3_app_web_behaviour.csv")
trading = load_table("table4_trading_activity.csv")
products = load_table("table5_product_subscription.csv")
demographics = load_table("table6_demographics_profile.csv")
support = load_table("table7_support_feedback.csv")
marketing = load_table("table8_marketing_engagement.csv")
target = load_table("table9_target_dormancy.csv")
metadata = load_table("table10_data_quality_metadata.csv")

print("Loaded tables:")
for name, df in [
    ("account", account), ("funding", funding), ("behaviour", behaviour),
    ("trading", trading), ("products", products), ("demographics", demographics),
    ("support", support), ("marketing", marketing), ("target", target),
    ("metadata", metadata)
]:
    print(f"{name:14s}: {df.shape[0]:>6,} rows × {df.shape[1]} columns")

target["observation_date"] = pd.to_datetime(target["observation_date"], errors="coerce")
target["dormant_within_6m"] = target["dormant_within_6m"].astype(str).str.strip().str.lower().map(
    {"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0}
)
target = target.dropna(subset=["customer_id", "dormant_within_6m"])
print("\nTarget distribution:")
print(target["dormant_within_6m"].value_counts().rename(index={0: "Not dormant", 1: "Dormant within 6 months"}))
print("Positive-class proportion:", round(target["dormant_within_6m"].mean(), 3))

Loaded tables:
account       :  3,227 rows × 11 columns
funding       : 10,515 rows × 11 columns
behaviour     : 13,064 rows × 11 columns
trading       : 11,614 rows × 12 columns
products      :  3,218 rows × 8 columns
demographics  :  3,218 rows × 6 columns
support       :  2,270 rows × 8 columns
marketing     :  3,218 rows × 10 columns
target        :  3,218 rows × 7 columns
metadata      : 14,387 rows × 5 columns

Target distribution:
dormant_within_6m
Not dormant                2197
Dormant within 6 months    1021
Name: count, dtype: int64
Positive-class proportion: 0.317
